# Challenge Notebook: Bruins Cold Brew Demand Forecasting

## Business Problem

Bruins Cold Brew & Bites is planning staffing and inventory for a series of alumni and campus events.
You have been given **daily historical demand data** and a few business context variables.

Your job is to build a lightweight forecasting workflow that mirrors how foundation models are often used in practice:

1. run a **zero-shot forecast** using a foundation-model style forecaster
2. compare it against **strong baselines**
3. **tune or calibrate** the forecast using recent validation data
4. re-evaluate on a held-out test period
5. make a deployment recommendation

## What you have

Dataset: `bruins_cold_brew_demand.csv` (at repo root or in `data/`)

Columns:
- `date`
- `avg_temp_c`
- `rain_mm`
- `campus_event`
- `promo_flag`
- `exam_period`
- `holiday_break`
- `is_weekend`
- `cups_sold`
- `split`

## Objective

Forecast daily `cups_sold` for the **test period**, then decide whether the forecasting workflow is strong enough to support operations.

## Suggested workflow

- **Part 1**: inspect the dataset and define the forecasting problem
- **Part 2**: generate zero-shot forecasts
- **Part 3**: evaluate against naive baselines
- **Part 4**: tune the zero-shot forecast using validation data
- **Part 5**: re-evaluate and make a deployment recommendation

This notebook is intentionally structured like a real-world analytics take-home: the data is ready, the business problem is concrete, and the focus is on judgment, not boilerplate.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except Exception:
    HAS_MPL = False

DATA_PATH = Path('bruins_cold_brew_demand.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('data/bruins_cold_brew_demand.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError('Could not locate bruins_cold_brew_demand.csv (try repo root or data/)')

CONFIG = {
    'context_len': 28,
    'seasonal_period': 7,
    'prefer_chronos': False,  # set True if Chronos is installed locally
}

CONFIG

## Part 1: Load and inspect the dataset

In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=['date'])
df.head()

In [ ]:
df.groupby('split').agg(
    rows=('cups_sold', 'size'),
    avg_cups=('cups_sold', 'mean'),
    avg_temp=('avg_temp_c', 'mean'),
    event_rate=('campus_event', 'mean'),
)

In [ ]:
if HAS_MPL:
    ax = df.plot(x='date', y='cups_sold', figsize=(12,4), title='Daily cold brew demand')
    ax.axvline(df.loc[df['split']=='validation', 'date'].min(), color='orange', linestyle='--', label='validation start')
    ax.axvline(df.loc[df['split']=='test', 'date'].min(), color='red', linestyle='--', label='test start')
    ax.legend()
    plt.tight_layout()
else:
    print(df[['date','cups_sold','split']].tail(12).to_string(index=False))

### Quick interpretation

Before modeling, answer these questions:

1. Is demand obviously seasonal?
2. Which business variables look likely to matter?
3. Why would a campus event or promo make this harder than a pure time-series problem?

In [ ]:
train_df = df[df['split']=='train'].reset_index(drop=True)
val_df = df[df['split']=='validation'].reset_index(drop=True)
test_df = df[df['split']=='test'].reset_index(drop=True)

print('train rows:', len(train_df), 'validation rows:', len(val_df), 'test rows:', len(test_df))

## Part 2: Create a zero-shot forecast

A realistic way to think about this step is:
- a pretrained time-series foundation model sees only the historical target series
- it produces a forecast without being trained specifically on this business

The notebook includes two paths:
- **Chronos path** if you have the open-source package installed
- **proxy foundation-model path** if you do not

The proxy path is here only so the exercise remains runnable on any laptop.

In [ ]:
def build_context_and_targets(full_df, context_len, forecast_split):
    start_idx = full_df.index[full_df['split'] == forecast_split][0]
    contexts, targets, feature_rows, target_dates = [], [], [], []
    for idx in range(start_idx, len(full_df)):
        if idx - context_len < 0:
            continue
        contexts.append(full_df.loc[idx-context_len:idx-1, 'cups_sold'].to_numpy(dtype=float))
        targets.append(float(full_df.loc[idx, 'cups_sold']))
        feature_rows.append(full_df.loc[idx, ['avg_temp_c','rain_mm','campus_event','promo_flag','exam_period','holiday_break','is_weekend']].to_numpy(dtype=float))
        target_dates.append(full_df.loc[idx, 'date'])
    return np.vstack(contexts), np.array(targets), np.vstack(feature_rows), pd.to_datetime(target_dates)

X_val_ctx, y_val, X_val_feat, val_dates = build_context_and_targets(df.reset_index(drop=True), CONFIG['context_len'], 'validation')
X_test_ctx, y_test, X_test_feat, test_dates = build_context_and_targets(df.reset_index(drop=True), CONFIG['context_len'], 'test')

X_val_ctx.shape, X_test_ctx.shape

In [ ]:
def try_chronos_one_step(contexts):
    try:
        import torch
        from chronos import ChronosPipeline
    except Exception as e:
        return None, {'used': False, 'path': 'chronos', 'reason': f'import failed: {type(e).__name__}: {e}'}

    try:
        pipeline = ChronosPipeline.from_pretrained('amazon/chronos-t5-small', device_map='cpu')
        preds = []
        for ctx in contexts:
            out = pipeline.predict(context=torch.tensor(ctx, dtype=torch.float32), prediction_length=1)
            arr = out.detach().cpu().numpy()
            pred = float(np.median(arr))
            preds.append(pred)
        return np.array(preds), {'used': True, 'path': 'chronos', 'model_id': 'amazon/chronos-t5-small'}
    except Exception as e:
        return None, {'used': False, 'path': 'chronos', 'reason': f'runtime failed: {type(e).__name__}: {e}'}


def proxy_foundation_forecast(contexts, seasonal_period=7):
    preds = []
    for ctx in contexts:
        x = np.asarray(ctx, dtype=float)
        recent = x[-7:]
        last = x[-1]
        seasonal = x[-seasonal_period] if len(x) >= seasonal_period else last
        trend = recent.mean() - x[-14:-7].mean() if len(x) >= 14 else 0.0
        pred = 0.55 * seasonal + 0.35 * recent.mean() + 0.10 * (last + trend)
        preds.append(pred)
    return np.array(preds), {'used': True, 'path': 'proxy_foundation'}

if CONFIG['prefer_chronos']:
    val_zero_pred, model_meta = try_chronos_one_step(X_val_ctx)
else:
    val_zero_pred, model_meta = (None, {'used': False, 'path': 'chronos', 'reason': 'disabled in CONFIG'})

if val_zero_pred is None:
    val_zero_pred, model_meta = proxy_foundation_forecast(X_val_ctx, CONFIG['seasonal_period'])

test_zero_pred, _ = proxy_foundation_forecast(X_test_ctx, CONFIG['seasonal_period']) if model_meta['path'] == 'proxy_foundation' else try_chronos_one_step(X_test_ctx)
print(model_meta)

## Part 3: Evaluate the zero-shot forecast against baselines

In [ ]:
def mae(y_true, y_pred):
    return float(np.mean(np.abs(y_true - y_pred)))

def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def smape(y_true, y_pred, eps=1e-6):
    denom = np.abs(y_true) + np.abs(y_pred) + eps
    return float(100 * np.mean(2 * np.abs(y_true - y_pred) / denom))


def last_value_baseline(contexts):
    return contexts[:, -1].astype(float)


def seasonal_naive_baseline(contexts, m=7):
    return contexts[:, -m].astype(float)


def score_row(name, y_true, y_pred):
    return {
        'model': name,
        'MAE': mae(y_true, y_pred),
        'RMSE': rmse(y_true, y_pred),
        'sMAPE': smape(y_true, y_pred),
    }

In [ ]:
val_last = last_value_baseline(X_val_ctx)
val_seasonal = seasonal_naive_baseline(X_val_ctx, CONFIG['seasonal_period'])

test_last = last_value_baseline(X_test_ctx)
test_seasonal = seasonal_naive_baseline(X_test_ctx, CONFIG['seasonal_period'])

validation_scores = pd.DataFrame([
    score_row('ZeroShot_Foundation', y_val, val_zero_pred),
    score_row('Baseline_LastValue', y_val, val_last),
    score_row('Baseline_SeasonalNaive', y_val, val_seasonal),
]).sort_values('RMSE').reset_index(drop=True)

validation_scores

In [ ]:
test_scores_zero = pd.DataFrame([
    score_row('ZeroShot_Foundation', y_test, test_zero_pred),
    score_row('Baseline_LastValue', y_test, test_last),
    score_row('Baseline_SeasonalNaive', y_test, test_seasonal),
]).sort_values('RMSE').reset_index(drop=True)

test_scores_zero

### Checkpoint

At this point students should answer:
- Does the zero-shot forecast actually add value?
- Is the gain large enough to matter operationally?
- If not, what would you try before giving up on the model?

## Part 4: Tune the forecast using validation data

Instead of full model fine-tuning, this challenge uses a lightweight but realistic pattern:

- keep the foundation forecast as the core signal
- fit a **calibration layer** on recent validation data
- use known business covariates (weather, events, promos) to correct systematic bias

This is common in production because it is fast, cheap, and often easier than retraining the whole model.

In [ ]:
def make_calibration_design(base_pred, features):
    base_pred = np.asarray(base_pred).reshape(-1, 1)
    feats = np.asarray(features, dtype=float)
    return np.hstack([
        np.ones((len(base_pred), 1)),
        base_pred,
        feats,
    ])


def fit_linear_calibrator(base_pred, features, y_true):
    X = make_calibration_design(base_pred, features)
    w, *_ = np.linalg.lstsq(X, y_true, rcond=None)
    return w


def apply_linear_calibrator(base_pred, features, weights):
    X = make_calibration_design(base_pred, features)
    return X @ weights

cal_weights = fit_linear_calibrator(val_zero_pred, X_val_feat, y_val)
val_tuned_pred = apply_linear_calibrator(val_zero_pred, X_val_feat, cal_weights)
test_tuned_pred = apply_linear_calibrator(test_zero_pred, X_test_feat, cal_weights)

cal_weights[:6]

In [ ]:
validation_scores_tuned = pd.DataFrame([
    score_row('ZeroShot_Foundation', y_val, val_zero_pred),
    score_row('Tuned_Foundation', y_val, val_tuned_pred),
    score_row('Baseline_LastValue', y_val, val_last),
    score_row('Baseline_SeasonalNaive', y_val, val_seasonal),
]).sort_values('RMSE').reset_index(drop=True)

validation_scores_tuned

In [ ]:
test_scores_tuned = pd.DataFrame([
    score_row('ZeroShot_Foundation', y_test, test_zero_pred),
    score_row('Tuned_Foundation', y_test, test_tuned_pred),
    score_row('Baseline_LastValue', y_test, test_last),
    score_row('Baseline_SeasonalNaive', y_test, test_seasonal),
]).sort_values('RMSE').reset_index(drop=True)

test_scores_tuned

In [ ]:
comparison_df = pd.DataFrame({
    'date': test_dates,
    'actual_cups': y_test,
    'zero_shot_pred': np.round(test_zero_pred, 1),
    'tuned_pred': np.round(test_tuned_pred, 1),
    'avg_temp_c': X_test_feat[:, 0],
    'campus_event': X_test_feat[:, 2].astype(int),
    'promo_flag': X_test_feat[:, 3].astype(int),
})
comparison_df.head(12)

In [ ]:
if HAS_MPL:
    plot_df = comparison_df.head(14)
    ax = plot_df.plot(x='date', y=['actual_cups','zero_shot_pred','tuned_pred'], figsize=(12,4), title='Test-period forecast comparison')
    plt.tight_layout()
else:
    print(comparison_df.head(14).to_string(index=False))

## Part 5: Student deliverable

Prepare a short recommendation memo answering the following:

1. Which model would you use for operations: zero-shot, tuned, or a baseline?
2. Did tuning help on the test set, or only on validation?
3. Would you deploy this workflow for the next alumni event?
4. What would you monitor after launch?

A strong answer should mention:
- at least one metric
- at least one baseline comparison
- one realistic failure mode
- one monitoring plan

In [ ]:
# Optional helper: deployment scorecard
best_test_model = test_scores_tuned.sort_values('RMSE').iloc[0]
rmse_best = float(best_test_model['RMSE'])
rmse_last = float(test_scores_tuned.loc[test_scores_tuned['model']=='Baseline_LastValue', 'RMSE'].iloc[0])
rmse_seasonal = float(test_scores_tuned.loc[test_scores_tuned['model']=='Baseline_SeasonalNaive', 'RMSE'].iloc[0])

scorecard = pd.DataFrame([
    {'check': 'Beats last-value baseline', 'result': rmse_best < rmse_last},
    {'check': 'Beats seasonal naive baseline', 'result': rmse_best < rmse_seasonal},
    {'check': 'RMSE below 15 cups/day', 'result': rmse_best < 15.0},
])
scorecard

## Final reflection

This challenge is meant to simulate a practical workflow:
- start with a foundation-model style forecast
- verify that it beats simple baselines
- improve it with a lightweight domain-specific tuning step
- decide whether the result is good enough to trust

That is much closer to real deployment work than simply training a model from scratch.